In [26]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import confusion_matrix, roc_auc_score


In [27]:
class RobustMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()


In [28]:
X_train = np.load("../artifacts/data/X_train.npy")
X_test  = np.load("../artifacts/data/X_test.npy")

y_train = np.load("../artifacts/data/y_train.npy")
y_test  = np.load("../artifacts/data/y_test.npy")


In [29]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_size = 194

stage1 = RobustMLP(input_size).to(device)
stage1.load_state_dict(
    torch.load("../artifacts/models/detector_stage1.pth", map_location=device)
)

stage1.eval()
#chamo os dados salvos no estágio 1

RobustMLP(
  (net): Sequential(
    (0): Linear(in_features=194, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.3, inplace=False)
    (6): Linear(in_features=64, out_features=1, bias=True)
  )
)

In [30]:
#detecção em cascata
#estágio 1: triagem
#não gasta mais tempo com tráfego claramente normal
#tráfego suspeito ou ataque são enviados para a segunda camada (especialista)

with torch.no_grad():
    logits = stage1(torch.tensor(X_train, dtype=torch.float32).to(device))
    probs = torch.sigmoid(logits).cpu().numpy()

threshold_stage1 = 0.5

suspect_mask = probs >= threshold_stage1

X_train_stage2 = X_train[suspect_mask]
y_train_stage2 = y_train[suspect_mask]

print("Samples para estágio 2:", X_train_stage2.shape) #quedra do número de amostras

#quero reduzir o FPR (refinar a detecção)

Samples para estágio 2: (114892, 194)


In [31]:
#diminuo a rede para 64 neurônios para entender apenas os nuances
#analisa evidências com mais vigor

class Stage2MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x).squeeze()


In [32]:
X2_t = torch.tensor(X_train_stage2, dtype=torch.float32)
y2_t = torch.tensor(y_train_stage2, dtype=torch.float32)

loader2 = DataLoader(
    TensorDataset(X2_t, y2_t),
    batch_size=256, #aumento o número de lotes, uma vez que o volume de dados é menor
    shuffle=True
)

model2 = Stage2MLP(X_train.shape[1]).to(device)

pos_weight_stage2 = torch.tensor([0.48]).to(device) #uso 0.5 pois quero um FPR menor aqui, uma vez que a maioria dos ataques já ficou no estágio 1
criterion2 = nn.BCEWithLogitsLoss(pos_weight=pos_weight_stage2)
optimizer2 = optim.Adam(model2.parameters(), lr=0.001)


In [33]:
epochs = 10

for epoch in range(epochs):
    total_loss = 0
    model2.train()

    for xb, yb in loader2:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer2.zero_grad()

        logits = model2(xb)
        loss = criterion2(logits, yb)

        loss.backward()
        optimizer2.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} - Loss: {total_loss/len(loader2):.4f}")


Epoch 1 - Loss: 0.0993
Epoch 2 - Loss: 0.0723
Epoch 3 - Loss: 0.0682
Epoch 4 - Loss: 0.0664
Epoch 5 - Loss: 0.0652
Epoch 6 - Loss: 0.0643
Epoch 7 - Loss: 0.0636
Epoch 8 - Loss: 0.0632
Epoch 9 - Loss: 0.0626
Epoch 10 - Loss: 0.0620


In [34]:
#filtragem dos estágios
#resultado: excelente equilíbrio no segundo estágio, com uma queda significativo do fpr para 16%
#com um custo de queda de 4% do Recall

stage1.eval()
model2.eval()

X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

with torch.no_grad():
    logits1 = stage1(X_test_t)
    probs1 = torch.sigmoid(logits1)

mask_stage1 = probs1 >= threshold_stage1

final_preds = np.zeros(len(X_test))

if mask_stage1.sum() > 0:
    X_stage2_test = X_test_t[mask_stage1]
    logits2 = model2(X_stage2_test)
    probs2 = torch.sigmoid(logits2).detach().cpu().numpy()

    final_preds[mask_stage1.cpu().numpy()] = (probs2 >= 0.5).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, final_preds).ravel()

recall = tp / (tp + fn)
fpr    = fp / (fp + tn)

print(f"Cascade Recall : {recall:.2%}")
print(f"Cascade FPR    : {fpr:.2%}")


Cascade Recall : 94.14%
Cascade FPR    : 16.35%
